# Understanding Neural Network Training Dynamics
## Hyperparameter Exploration with Fashion-MNIST

**Course:** Emerging Topics in Data Analytics & Management  
**Institution:** IE University — Bachelor in Data & Business Analytics  
**Academic Year:** 2025–2026

---

**Student Name:**  Luisa Johanna Kaczmarek
**Student ID:**  16242
**Date:** 03.03.2026

---

> **Instructions:** Work through each section sequentially. Run all experiments, fill in the results tables,
> create the required plots, and write your analysis in the designated markdown cells.
> Do not modify the data preprocessing code. Change only one hyperparameter at a time during experiments.


## 0. Environment Setup & Imports

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

# Verify GPU availability (recommended but not required)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Set random seeds for partial reproducibility
# Note: GPU operations are not fully deterministic, so results will vary slightly between runs
tf.random.set_seed(42)
np.random.seed(42)


---
## 1. Data Loading & Preprocessing

The following cell loads Fashion-MNIST and prepares the train/validation/test splits.  
**Do not modify this cell** — consistent preprocessing is required for meaningful comparisons.


In [ ]:
# Load Fashion-MNIST
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Normalize pixel values to [0, 1]
X_train_full = X_train_full.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Split training into train (50k) and validation (10k)
X_train, X_val = X_train_full[:50000], X_train_full[50000:]
y_train, y_val = y_train_full[:50000], y_train_full[50000:]

# Class names
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Training set:   {X_train.shape[0]:,} samples, shape: {X_train.shape[1:]}")
print(f"Validation set: {X_val.shape[0]:,} samples")
print(f"Test set:       {X_test.shape[0]:,} samples")
print(f"Number of classes: {len(class_names)}")
print(f"Pixel value range: [{X_train.min()}, {X_train.max()}]")


- image size: 28 x 28 (after normalisation scaling)
- multi-class -> 10 classes -> our output vector will have size 10 (after softmax activatino) -> this vetcor will contain probabilities of each class, adding up to 1

---
## 2. Data Exploration

Before running any experiments, explore the dataset visually.


### 2.1 Sample Images
Display at least 2 sample images per class (20 images total) in a grid.


In [ ]:
# TODO: Create a grid showing 2 sample images per class (2 rows x 10 columns or 4 rows x 5 columns)
# Hint: Use plt.subplot() and iterate over class_names

fig, axes = plt.subplots(2, 10, figsize=(20, 5))
fig.suptitle('Sample Images from Each Class', fontsize=14, fontweight='bold')

for class_idx in range(10):
    # Find indices of samples belonging to this class
    class_indices = np.where(y_train == class_idx)[0]

    for row in range(2):
        ax = axes[row, class_idx]
        ax.imshow(X_train[class_indices[row]], cmap='gray')
        if row == 0:
            ax.set_title(class_names[class_idx], fontsize=8)
        ax.axis('off')

plt.tight_layout()
plt.show()


### 2.2 Pixel Intensity Distribution
Plot a histogram of pixel values across the training set.


In [ ]:
# TODO: Plot histogram of pixel intensities
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(X_train.flatten(), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Pixel Intensity')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Pixel Intensities (Training Set)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


-->  this distribution tells us that most pixels are black (Pixel intensity=0), then we have very few pixels in dark grey, and then a bigger group of pixels that are very light grey (Pixel intensity slightly over 0.8. Then, we also see a group of white pixels, however, in terms of quantity we only have approx 1/32 white pixels (Pixel intensity=1) in comparison to the number of black pixels (Pixel intensity=0).

### 2.3 Your Observations

**Visually similar pairs likely to confuse a classifier:**

- **T-shirt/top vs. Shirt** — both are upper-body garments with similar silhouettes (short sleeves, collar region, roughly symmetric torso shape); the distinction often lies in subtle texture or neckline details that 28×28 resolution compresses aggressively.
- **Pullover vs. Coat** — both are long-sleeved, center-symmetric tops; coats tend to have slightly more defined lapels or length, but at this resolution that signal is weak and noisy.
- **Sandal vs. Ankle boot** — both are footwear oriented horizontally, but sandals show open lattice-like structure while boots are solid. The second-row sandal sample (appears mostly white/bright) already looks quite boot-like.
- **Sneaker vs. Ankle boot** — thick sole, similar bounding-box aspect ratio, both solid; distinguishing them relies on upper height, which is a fragile pixel-level cue.

The core difficulty is that Fashion-MNIST's low resolution (28×28, 8-bit grayscale) destroys fine details — stitching, fabric texture, collar shape — that a human uses effortlessly. Any classifier that doesn't learn global silhouette *and* local texture jointly will conflate the upper-body garment cluster (T-shirt, Shirt, Pullover, Coat) and the closed-toe footwear cluster (Sneaker, Ankle boot) at a higher rate than the other classes.


---
## 3. Plotting Utility

The following helper function will be used throughout the exercise. You may customize its appearance, but keep the same information (training & validation loss and accuracy).


In [ ]:
def plot_history(history, title='Training History'):
    """Plot training and validation loss/accuracy curves side by side."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    ax1.plot(history.history['loss'], label='Training Loss', linewidth=2)
    ax1.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} — Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Accuracy
    ax2.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
    ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title(f'{title} — Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_overlay(histories, metric='val_loss', title='Comparison'):
    """Overlay a specific metric from multiple training histories.

    Args:
        histories: dict of {label: history_object}
        metric: which metric to plot (e.g., 'val_loss', 'val_accuracy')
        title: plot title
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    for label, history in histories.items():
        ax.plot(history.history[metric], label=label, linewidth=2)

    ax.set_xlabel('Epoch')
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def summarize_history(history):
    """Print a quick summary of final metrics from a training history."""
    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    best_val_epoch = np.argmin(history.history['val_loss']) + 1

    print(f"  Final Train Loss:     {final_train_loss:.4f}")
    print(f"  Final Val Loss:       {final_val_loss:.4f}")
    print(f"  Final Train Accuracy: {final_train_acc:.4f}")
    print(f"  Final Val Accuracy:   {final_val_acc:.4f}")
    print(f"  Train-Val Acc Gap:    {final_train_acc - final_val_acc:.4f}")
    print(f"  Best Val Loss Epoch:  {best_val_epoch}")
    return {
        'train_loss': final_train_loss,
        'val_loss': final_val_loss,
        'train_acc': final_train_acc,
        'val_acc': final_val_acc,
        'gap': final_train_acc - final_val_acc,
        'best_epoch': best_val_epoch
    }


---
## 4. Part 1: Establishing a Baseline

Build the specified baseline model. All subsequent experiments will be compared against this.

**Baseline configuration:**
- Architecture: Flatten → Dense(128, relu) → Dense(64, relu) → Dense(10, softmax)
- Optimizer: Adam (lr=0.001)
- Batch size: 64
- Epochs: 30


In [ ]:
def build_baseline():
    model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

baseline = build_baseline()
baseline.summary()


- input size is the normalised image size (28x28) 'Flatten' -0> we trun it into a flat vectoor/ 1-dim (because we deisgn a normla ANN here, nota. CNN< so ot has to be a flattened vector
- first hidden layer has 128
- 100480 is how we connect input vector of size 784 with 1st dense vectir of size 128
- Trainable params -> the wieghts. the ones that are trained.


model.compile:
- we choose adam parameter with learning rate 0.001.
- loss function: sparse categoriclal corss entropy.
- metric: accuracy

In [ ]:
history_baseline = baseline.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    verbose=1
)

plot_history(history_baseline, 'Baseline Model')
print("\n--- Baseline Results ---")
baseline_results = summarize_history(history_baseline)


### Baseline Interpretation (ANN)

- **left plot (loss):** clear overfitting — training loss drops continuously to 0.13 while validation loss bottoms out around epoch 7 (~0.34) and then climbs steadily to 0.47. The gap of ~0.34 by epoch 30 is a textbook divergence signal. Best val loss epoch (7) is confirmed in the printed results.

- **right plot (accuracy):** overfitting is much less dramatic here — validation accuracy plateaus around 0.88 after epoch ~5 and barely moves for the remaining 25 epochs, while training accuracy keeps climbing to 0.95. The 7pp train-val gap is notable but the validation curve isn't degrading, just stalling.

- **takeaway:** loss and accuracy tell slightly different stories — loss says the model is actively getting worse on unseen data after epoch 7, while accuracy says it's just stopped learning anything useful. Both point to the same fix: early **stopping around epoch 6–8,** which would recover ~0.34 val loss instead of the final 0.47.


---
## 5. Part 2: Hyperparameter Experiments

> **Reminder:** Change only ONE hyperparameter at a time. All other settings remain at baseline values.

### Experiment A: Learning Rate

The learning rate controls the step size during gradient descent. You will test five values spanning several orders of magnitude.


In [ ]:
learning_rates = [0.1, 0.01, 0.001, 0.0001, 0.00001]
lr_histories = {}

for lr in learning_rates:
    print(f'\n{"="*60}')
    print(f'Training with learning rate = {lr}')
    print(f'{"="*60}')

    model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train, y_train,
        epochs=30,
        batch_size=64,
        validation_data=(X_val, y_val),
        verbose=0
    )

    lr_histories[lr] = history
    plot_history(history, f'Learning Rate = {lr}')
    summarize_history(history)


**Observations:**

Learning rate 0.1:
- large oscillations in both loss and accuracy — the optimizer is overshooting the minimum and unable to settle
- final val accuracy is 0.096 and val loss is 2.31 — essentially random performance for a 10-class problem (random baseline = 0.10), making this LR completely unusable

Learning rate 0.01:
- loss: validation loss is consistently higher than training loss, noisy, and trending upward — a clear sign of overfitting. The instability (epoch-to-epoch oscillation) also suggests the LR is still too large for stable fine-grained updates
- accuracy: validation accuracy plateaus around 0.87, which is a reasonable value, but the lack of full stabilisation paired with the diverging loss confirms the model is not generalising well

Learning rate 0.001:
- training loss keeps dropping to 0.12 while validation loss bottoms around epoch 7 (0.34) and then climbs to 0.51 — the same overfitting pattern as the baseline, just smoother
- accuracy: validation accuracy stabilises around 0.875, best val loss epoch is 7, meaning anything trained beyond that is overfitting
- verdict: strong training performance but clear overfitting; should be paired with early stopping at epoch 7

Learning rate 0.0001:
- train and validation curves stay close together throughout — the least overfitting of all five LRs, and the best final val accuracy (0.8805) and lowest final val loss (0.3331)
- neither curve has fully converged by epoch 30, meaning the model is still improving; best val loss epoch was 29
- verdict: most stable and best final numbers within this budget, but needs more epochs to fully converge


#### Overlay Plot: All Learning Rates

In [ ]:
# Overlay validation loss for all learning rates
lr_labeled = {f'lr={lr}': h for lr, h in lr_histories.items()}
plot_overlay(lr_labeled, metric='val_loss', title='Learning Rate Comparison — Validation Loss')
plot_overlay(lr_labeled, metric='val_accuracy', title='Learning Rate Comparison — Validation Accuracy')


#### Analysis A

**A1 — lr=0.1:** Describe the training curve. Does the model converge? If not, explain why in terms of gradient descent mechanics.

- The model does not converge, as:
    - Validation loss:
        - Spikes immediately and oscillates at around 2.3 — more than double the loss of any other LR.
        - Does not stabilize at a low value; best val loss is at epoch 1.
    - Validation accuracy:
        - Final val accuracy is 0.096 — statistically indistinguishable from random guessing on a 10-class problem.
        - Remains flat and extremely low throughout.
-> Learning rate is too large — the optimizer overshoots the minimum on every step and never recovers.

---

**A2 — lr=0.01 vs lr=0.001:** Which converges faster? Which achieves lower final validation loss? Is faster convergence always better?

- lr=0.01 converges faster — it reaches stable validation accuracy by around epoch 5, while lr=0.001 takes longer to settle and still shows a slight upward trend in val loss through epoch 30.

- lr=0.01 also achieves the lower final validation loss: **0.4418 vs 0.5068** for lr=0.001. On accuracy they are close (0.870 vs 0.875), but lr=0.001 edges ahead slightly in accuracy while lr=0.01 is better on loss — reflecting that lr=0.001 is overfitting more aggressively in the later epochs.

- Is faster convergence always better? No:
    - lr=0.1 converges "instantly" but to a random-guessing plateau — speed meant failure.
    - lr=0.01 converges fast and reaches reasonable accuracy, but its loss trends upward after epoch 6, hinting that the larger steps prevent it from settling into a stable minimum.
    - lr=0.001 is slower early on but continues improving accuracy; paired with early stopping at epoch 7 it recovers a val loss of ~0.34 rather than the final 0.51.

---

**A3 — lr=0.00001:** How does the model behave over 30 epochs? Would more epochs help? What is the trade-off?

-> The purple line tells a classic slow learner story:

- Accuracy: starts low and climbs steadily, reaching 0.849 by epoch 29. The curve still has a slight upward tilt at the end — not yet plateaued.
- Loss: starts high and drops continuously, finishing around 0.43. The curve is smooth with no oscillation at all.

- Key observation: at epoch 29 it hasn't converged yet. Unlike lr=0.001 which flattened and then overfit, this curve is still descending. More epochs would certainly help.

- The trade-off: the time and compute cost to reach the same performance as lr=0.001 (which does it in ~7 epochs) makes it inefficient unless training budget is unconstrained.

---

**A4 — Recommendation:** Which learning rate would you recommend for this problem and why?

I would recommend lr=0.0001 — it achieves the lowest final validation loss (0.3331) and the highest final validation accuracy (0.8805) of all five learning rates, with a smooth, stable curve showing no oscillation or overfitting within 30 epochs.

That said, this recommendation is budget-conditional. Within 30 epochs, lr=0.0001 wins on stability and final metrics. If training budget were tight, lr=0.001 with early stopping at epoch 7 would recover a val loss of ~0.34 at a fraction of the compute cost — so the 'best' LR depends on what you're optimising for.


---
### Experiment B: Batch Size

Batch size affects gradient noise, training speed, and generalization. You will test five values while timing each run.


In [ ]:
batch_sizes = [16, 32, 64, 256, 1024]
bs_histories = {}
bs_times = {}

for bs in batch_sizes:
    print(f'\n{"="*60}')
    print(f'Training with batch size = {bs}')
    print(f'{"="*60}')

    model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    start = time.time()
    history = model.fit(
        X_train, y_train,
        epochs=30,
        batch_size=bs,
        validation_data=(X_val, y_val),
        verbose=0
    )
    elapsed = time.time() - start

    bs_histories[bs] = history
    bs_times[bs] = elapsed / 30  # average seconds per epoch

    print(f'  Average time per epoch: {elapsed/30:.2f}s')
    plot_history(history, f'Batch Size = {bs}')
    summarize_history(history)


#### Overlay Plot: All Batch Sizes

In [ ]:
# Overlay validation loss for all batch sizes
bs_labeled = {f'bs={bs}': h for bs, h in bs_histories.items()}
plot_overlay(bs_labeled, metric='val_loss', title='Batch Size Comparison — Validation Loss')
plot_overlay(bs_labeled, metric='val_accuracy', title='Batch Size Comparison — Validation Accuracy')


#### Timing Analysis

In [ ]:
# Bar chart: average time per epoch for each batch size
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([str(bs) for bs in batch_sizes], [bs_times[bs] for bs in batch_sizes],
       color='steelblue', edgecolor='white')
ax.set_xlabel('Batch Size')
ax.set_ylabel('Average Time per Epoch (seconds)')
ax.set_title('Training Speed vs. Batch Size')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, bs in enumerate(batch_sizes):
    ax.text(i, bs_times[bs] + 0.02, f'{bs_times[bs]:.2f}s', ha='center', fontsize=10)

plt.tight_layout()
plt.show()


#### Analysis B

**B1 — bs=16 vs bs=1024:** Which shows more noise in the loss curve? Which achieves better final validation accuracy? Explain why, referencing gradient noise as implicit regularization.

- bs=16 (blue) is far noisier — the validation loss oscillates heavily and climbs to 0.60 by epoch 30. bs=1024 (purple) is the smoothest curve of all five, ending around 0.33–0.35.
- On validation accuracy though, they end up nearly tied (0.88), which is the interesting part.
- Why: small batches compute gradients from very few samples each step, so each gradient is a noisy estimate of the true gradient. That noise acts as implicit regularization — it prevents the model from settling into sharp minima and can help generalization. But past a point it stops helping and just destabilizes training, which is what the loss curve shows. bs=1024 gets clean, low-variance gradient estimates each step, which is why the curve is smooth — but those sharp minima it finds don't always generalize better, hence the similar final accuracy.

---

**B2 — Speed vs. accuracy trade-off:** Describe the relationship between batch size and training time. Why does this relationship exist?

- The relationship is clear and steep: bs=16 takes 9.27s/epoch, bs=32 takes 4.58s, bs=64 takes 2.32s, bs=256 takes 0.81s, bs=1024 takes 0.44s.
- Roughly, doubling the batch size halves the time — but not exactly, because GPU parallelism means large batches don't cost proportionally more compute, they just saturate the hardware more efficiently.
- With small batches, you do many more weight updates per epoch (more forward/backward passes), each of which has fixed overhead. Large batches amortize that overhead across more samples per pass, which is why the speedup is so dramatic going from 16 to 256.

---

**B3 — Sweet spot:** Is there an optimal batch size for this problem? Justify considering both accuracy and efficiency.

- bs=1024 is the strongest result within this 30-epoch window — lowest val loss (0.33–0.35), equivalent accuracy to all other sizes (0.88), and the fastest training at 0.44s/epoch. The smooth, still-descending curve suggests it hasn't overfit and could improve further with more epochs.
- bs=256 is a reasonable second choice: stable val loss (0.38), same accuracy, and 0.81s/epoch — still 11x faster than bs=16. The only genuine argument for preferring it over 1024 is theoretical (large batches finding sharper minima), but the charts here don't actually show that happening.
- bs=16 and bs=32 pay a large time penalty for noisy curves that don't translate into better final accuracy — hard to justify.
- Bottom line: bs=1024 wins on every measurable axis within 30 epochs. If training budget were tight or the dataset were much larger, the sharp-minima concern would be worth revisiting — but not here.

---
### Experiment C: Optimizer Selection

Different optimizers navigate the loss landscape differently. You will compare four common optimizers.


In [ ]:
optimizer_configs = {
    'SGD': tf.keras.optimizers.SGD(learning_rate=0.001),
    'SGD+Momentum': tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9),
    'RMSprop': tf.keras.optimizers.RMSprop(learning_rate=0.001),
    'Adam': tf.keras.optimizers.Adam(learning_rate=0.001),
}
opt_histories = {}

for name, opt in optimizer_configs.items():
    print(f'\n{"="*60}')
    print(f'Training with optimizer: {name}')
    print(f'{"="*60}')

    model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=opt,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train, y_train,
        epochs=30,
        batch_size=64,
        validation_data=(X_val, y_val),
        verbose=0
    )

    opt_histories[name] = history
    plot_history(history, f'Optimizer: {name}')
    summarize_history(history)


#### Overlay Plot: All Optimizers

In [ ]:
plot_overlay(opt_histories, metric='val_loss', title='Optimizer Comparison — Validation Loss')
plot_overlay(opt_histories, metric='val_accuracy', title='Optimizer Comparison — Validation Accuracy')


#### Analysis C

**C1 — Overall comparison:** Which optimizer converges fastest? Which achieves the lowest final validation loss?

- Fastest convergence: Adam and RMSprop — both drop to ~0.35 val loss within the first 3–5 epochs, far ahead of SGD variants at that point.
- Lowest final val loss: SGD+Momentum wins clearly at 0.3520, and crucially it's still descending at epoch 30 with no sign of divergence. Adam ends at 0.4671, RMSprop at 0.5537, plain SGD at 0.4859.
- The comparison chart makes this stark — orange (SGD+Momentum) is the only line that keeps going down while everything else either flattens or drifts upward. This is consistent with the sharp-minima hypothesis (Keskar et al. 2017): adaptive optimisers may converge to sharper minima that generalise less well, though the evidence here is suggestive rather than definitive.

---

**C2 — SGD vs SGD+Momentum:** How does momentum change the training dynamics? Reference the shape and smoothness of the curves.

- Plain SGD is slow and smooth — val loss starts at 1.4 and descends steadily but hasn't converged by epoch 30. Val accuracy only reaches 0.8319, the lowest of all four optimizers.
- Adding momentum dramatically changes both speed and final performance — val loss drops much faster in the early epochs, the curve is still smooth (no oscillation), and it ends at 0.3520 val loss with 0.8751 val accuracy.
- Mechanically: momentum accumulates a running average of past gradients, so the optimizer builds up speed in consistent directions and dampens oscillation in noisy ones. The result is faster progress down the loss landscape without the instability you'd get from just raising the learning rate.

---

**C3 — Adam vs RMSprop:** Both use adaptive learning rates. Do they perform similarly? If there are differences, hypothesize why.

- Early training: nearly identical — both converge fast to ~0.35 val loss by epoch 5.
- After that they diverge: RMSprop val loss climbs to 0.5537 by epoch 30 (Best Val Epoch: 9), while Adam is more stable, ending at 0.4671 (Best Val Epoch: 9). They share the same best val epoch, but RMSprop deteriorates more steeply after it.
- On accuracy they're close — RMSprop 0.8782, Adam 0.8844 — so loss is again the more informative signal.
- Why the difference: Adam adds first-moment estimation (momentum) on top of RMSprop's second-moment scaling, which helps smooth out noisy updates and resist overfitting longer. RMSprop without momentum can take increasingly large steps in directions with low recent gradient variance, which likely explains the steeper loss increase after epoch 9.

---

**C4 — When might SGD be preferred over Adam?**

- SGD (especially with momentum) is often preferred when generalization is the priority over convergence speed. The literature on sharp vs flat minima suggests that adaptive optimizers like Adam tend to find sharper minima — regions where small input changes cause large loss changes — which can hurt test performance on unseen distributions.
- This experiment is consistent with that pattern: SGD+Momentum achieves the lowest val loss of all four (0.3520), even though Adam trains faster and reaches lower training loss (0.1320 vs 0.3048). The sharp-minima hypothesis remains contested in the literature, but these results align with it.
- In practice, SGD+Momentum is still the default in computer vision precisely because the slower, noisier updates act as implicit regularization and tend to generalize better. Adam is preferred when you need fast convergence, the loss landscape is complicated, or you're working on NLP/transformers where adaptive rates matter more.


---
### Experiment D: Architecture (Depth and Width)

Network capacity — determined by depth (number of layers) and width (neurons per layer) — controls the range of functions the model can represent.


In [ ]:
architectures = {
    'D1_minimal':      [32],
    'D2_shallow_wide': [512],
    'D3_baseline':     [128, 64],
    'D4_deep_narrow':  [64, 64, 64, 64],
    'D5_deep_wide':    [256, 256, 128, 64],
    'D6_very_deep':    [128, 128, 128, 128, 128, 128],
}
arch_histories = {}
arch_params = {}

for name, layers in architectures.items():
    print(f'\n{"="*60}')
    print(f'Training {name}: hidden layers = {layers}')
    print(f'{"="*60}')

    model = tf.keras.Sequential([tf.keras.layers.Flatten(input_shape=(28, 28))])
    for units in layers:
        model.add(tf.keras.layers.Dense(units, activation='relu'))
    model.add(tf.keras.layers.Dense(10, activation='softmax'))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    total_params = model.count_params()
    arch_params[name] = total_params
    print(f'  Total parameters: {total_params:,}')

    history = model.fit(
        X_train, y_train,
        epochs=30,
        batch_size=64,
        validation_data=(X_val, y_val),
        verbose=0
    )

    arch_histories[name] = history
    plot_history(history, f'Architecture: {name}')
    summarize_history(history)


#### Overlay Plot: All Architectures

In [ ]:
plot_overlay(arch_histories, metric='val_loss', title='Architecture Comparison — Validation Loss')
plot_overlay(arch_histories, metric='val_accuracy', title='Architecture Comparison — Validation Accuracy')


#### Parameter Count vs. Validation Accuracy

In [ ]:
# Scatter plot: total parameters vs final validation accuracy
fig, ax = plt.subplots(figsize=(10, 6))

for name in architectures:
    params = arch_params[name]
    val_acc = arch_histories[name].history['val_accuracy'][-1]
    ax.scatter(params, val_acc, s=120, zorder=5)
    ax.annotate(name, (params, val_acc), textcoords="offset points",
                xytext=(10, 5), fontsize=9)

ax.set_xlabel('Total Parameters')
ax.set_ylabel('Final Validation Accuracy')
ax.set_title('Model Capacity vs. Validation Accuracy')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


#### Analysis D

**D1 — D1 (minimal) vs D5 (deep-wide):** Which shows underfitting? Which shows overfitting? How can you tell from the training curves?

- D1_minimal (underfitting): only 25k parameters and one layer of 32 units. Training loss only drops to 0.2378 and val loss stays close at 0.3777 — the gap is the smallest of all architectures (0.0384). The model is too small to memorize training data, which also means it's too small to learn the full complexity of the task. Val accuracy caps at 0.8762, the lowest of all six.
- D5_deep_wide (overfitting): train loss drops to 0.1215 while val loss climbs to 0.5043 — one of the largest divergences in the set. The loss curves start separating around epoch 5 and the gap keeps widening. Best val epoch was 8, meaning the last 22 epochs  of training were actively making generalization worse

---

**D2 — D2 (shallow-wide) vs D4 (deep-narrow):** Both have a similar parameter count. Which performs better? What does this tell you about depth vs. width?

- Worth flagging: the parameter counts aren't actually similar — D2 has 407k and D4 has 63k, so this is more of a depth vs. width comparison than a controlled one.
- D2 (shallow, one wide layer of 512): val accuracy 0.8819, but heavy overfitting — val loss climbs to 0.5081 with best val epoch at 8.
- D4 (deep, four narrow layers of 64): val accuracy 0.8803, val loss 0.4453, slightly less overfitting despite being deeper.
- Neither is clearly better on accuracy, but D4 uses 6x fewer parameters to get nearly the same result. This suggests depth extracts more useful representations per parameter than simply widening a single layer — width alone without depth doesn't help generalization here.

---

**D3 — D6 (very deep):** Do you observe training difficulties such as vanishing gradients or slow convergence?

- No obvious vanishing gradient symptoms — training loss decreases steadily from 0.55 to 0.15 over 30 epochs, meaning gradients are flowing through all 6 layers reasonably well.
- What you do see is noisy val loss — it oscillates significantly epoch-to-epoch, suggesting the deep network is sensitive to batch-level variation, possibly because the many layers amplify small weight perturbations.
- D6 ends with epoch-30 val accuracy 0.8771 and val loss 0.4997; its **peak** val accuracy across all epochs was 0.8891 (the highest of all six architectures), which is why it is selected as the best ANN for the CNN comparison in Part 3. D5_deep_wide had the highest **final** (epoch-30) val accuracy at 0.8863 — the scatter plot reflects this epoch-30 snapshot. D4_deep_narrow achieves the second-lowest val loss (0.4453) with a fraction of the parameters. Depth alone does not guarantee better performance; the combination of depth and width in D5 drives the epoch-30 edge, while D6's added depth delivers a higher peak accuracy before overfitting increases.

---

**D4 — Parameter count vs. accuracy:** Is there a clear relationship in the scatter plot?

- No clear relationship — the scatter plot makes this obvious. D2_shallow_wide has the most parameters (407k) but a middling val accuracy (0.8819). D5_deep_wide achieves the best val accuracy (0.8863) with 308k parameters, while D1_minimal has the fewest params and the lowest accuracy — but beyond that minimum threshold, more parameters don't consistently help.
- The takeaway: architecture shape matters more than raw parameter count. D4 achieves 0.8803 accuracy with only 63k parameters, outperforming D2 which uses 6x more. Fashion-MNIST rewards hierarchical feature extraction over brute-force capacity.


---
## 6. Part 3: From ANN to CNN

Fashion-MNIST images have spatial structure. A CNN exploits this through learned convolutional filters. Build the specified CNN and compare it against your best ANN.


In [ ]:
def build_cnn():
    model = tf.keras.Sequential([
        tf.keras.layers.Reshape((28, 28, 1), input_shape=(28, 28)),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn_model = build_cnn()
cnn_model.summary()


In [ ]:
history_cnn = cnn_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    verbose=1
)

plot_history(history_cnn, 'CNN Baseline')
print("\n--- CNN Results ---")
cnn_results = summarize_history(history_cnn)


#### ANN vs CNN Comparison Plot

In [ ]:
# Best ANN from architecture experiments: D6_very_deep [128, 128, 128, 128, 128, 128]
# Highest val accuracy (0.8891) and second-lowest val loss (0.4333) across all six architectures
best_ann_history = arch_histories['D6_very_deep']

comparison = {
    'Best ANN (D6_very_deep)': best_ann_history,
    'CNN': history_cnn,
}

plot_overlay(comparison, metric='val_loss', title='Best ANN vs CNN — Validation Loss')
plot_overlay(comparison, metric='val_accuracy', title='Best ANN vs CNN — Validation Accuracy')

#### Analysis: ANN vs CNN

**1.** Compare the total parameter count of the CNN with your best ANN. Which has more parameters? Despite this, which achieves better validation accuracy? What does this suggest about inductive bias?

- D6_very_deep has 184k parameters. The CNN (Conv32→Pool→Conv64→Pool→Dense128→Dense10) has **~422k trainable parameters** — significantly more than D6. (Conv2D(32,3×3): 320; Conv2D(64,3×3): 18,496; Dense(128) from 7×7×64=3,136 features: 401,536; Dense(10): 1,290 → total ≈ 421,642.)
- Despite having more parameters, the CNN achieves better val accuracy (~0.91) vs the ANN's 0.889.
- This is a powerful demonstration of inductive bias: the CNN doesn't need more parameters *efficiently deployed* to win — it wins because its architecture encodes prior knowledge about images — locality, spatial hierarchy, shared filters. The ANN has no such prior and must learn all spatial relationships through fully connected weights across all 784 pixels simultaneously, which is both less efficient and harder to generalise from. Architecture, not parameter count, is the decisive factor.

---

**2.** Comment on the convergence speed and final performance based on the overlay plot.

- CNN converges faster and higher — it reaches ~0.91 val accuracy by epoch 3 and stays there, while the ANN is still climbing at epoch 30, ending around 0.889.
- On val loss though, the CNN overfits badly — it bottoms out at epoch 4 (0.23) and then climbs steeply to 0.64 by epoch 30. The ANN's val loss is more stable, staying in the 0.35–0.45 range throughout.
- So the CNN wins on accuracy but is a worse-behaved model in terms of loss — it would strongly benefit from early stopping at epoch 4.

---

**3.** Why are CNNs more suitable than ANNs for image data?

CNNs use local receptive fields — each filter only looks at a small spatial patch at a time — which matches the structure of images, where nearby pixels are more semantically related than distant ones. Parameter sharing means the same filter is applied across the entire image, so the network learns to detect a feature (like an edge or texture) regardless of where it appears, with a fraction of the parameters a fully connected layer would need. This directly gives translational equivariance: if a shoe shifts two pixels to the right, the CNN's activations shift accordingly rather than treating it as a completely new input. An ANN flattens the image and loses all spatial structure immediately, forcing it to relearn every sp

### 6.1 Optional Extension: CNN Hyperparameter Sensitivity

If time allows, pick one hyperparameter from Part 2 and repeat that experiment with the CNN architecture. Do CNNs show the same sensitivity?


In [ ]:
### 6.1 Optional Extension: CNN Hyperparameter Sensitivity

cnn_lr_histories = {}
for lr in [0.01, 0.001, 0.0001]:
    model = build_cnn()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    history = model.fit(
        X_train, y_train,
        epochs=30,
        batch_size=64,
        validation_data=(X_val, y_val),
        verbose=0
    )
    cnn_lr_histories[f'CNN lr={lr}'] = history
    print(f"lr={lr} — best val loss: {min(history.history['val_loss']):.4f} "
          f"at epoch {history.history['val_loss'].index(min(history.history['val_loss'])) + 1}")

plot_overlay(cnn_lr_histories, metric='val_loss', title='CNN — Learning Rate Sensitivity (Val Loss)')
plot_overlay(cnn_lr_histories, metric='val_accuracy', title='CNN — Learning Rate Sensitivity (Val Accuracy)')

#### Optional Extension Analysis

**CNN LR Sensitivity — do CNNs show the same sensitivity?**

Short answer: yes, the same directional patterns hold, but the CNN is more robust and reaches better absolute performance at every LR.

- lr=0.01: overfits fast — val loss bottoms at 0.304 at epoch 2 then climbs to 0.65+. Same instability as the ANN, but the CNN still hits 0.88 val accuracy, which the ANN at lr=0.01 also managed. The pattern is identical, the damage is similar.

- lr=0.001: best val loss at epoch 4 (0.252), then diverges upward — same early-then-overfit story as the ANN. But val accuracy holds at 0.91 through epoch 30, noticeably better than the ANN's 0.876 at the same Lr

- lr=0.0001: still descending at epoch 30 (val loss 0.246, still going down), val accuracy climbing to 0.915 and not yet plateaued — exactly the "slow but not converged" story from the ANN. The key difference: the CNN at lr=0.0001 is *still improving* and already matches lr=0.001 on accuracy, whereas the ANN at lr=0.0001 was trailing behind and needed significantly more epochs to catch up.

**What's different vs ANN:**
- The CNN is less sensitive in the sense that all three LRs reach 0.88–0.91 accuracy — the ANN had a much wider spread. The CNN's inductive bias (parameter sharing, local filters) gives it a stronger starting point that partially compensates for suboptimal LR choices.
- The overfitting pattern (loss diverging after best val epoch) is actually *worse* in the CNN — it overfits faster and more severely, consistent with what we saw in the baseline CNN analysis. The CNN learns faster, but that also means it memorizes faster.

**Recommendation:** lr=0.001 with early stopping at epoch 4 is the clear choice — lowest val loss, highest accuracy, and clean convergence before the overfit kicks in.


---
## 7. Part 4: Best Model on Test Set

Based on your experiments, select your best hyperparameter combination. Train it and evaluate on the test set **exactly once**.

### Your Final Configuration


| Decision | Choice | Justification |
|----------|--------|---------------|
| Architecture type | **CNN** | Best val accuracy of the entire experiment: ~0.91 at early stopping epoch, vs 0.889 for best ANN (D6_very_deep). Inductive bias for spatial data justifies it unambiguously. |
| Hidden layers | Conv2D(32) → Pool → Conv2D(64) → Pool → Dense(128) | The specified CNN architecture; fully justified in Part 3 analysis |
| Learning rate | **0.001** | From the CNN LR sensitivity extension: lr=0.001 achieves the lowest val loss (0.252 at epoch 4) before overfitting. For the CNN specifically, 0.001 beats 0.0001 within a short epoch budget. |
| Batch size | **256** | bs=1024 was fastest, but for the CNN we want stable gradients at the cost of minimal speed overhead. bs=256 delivers low val loss (0.38 in ANN experiments) with reasonable noise, and the CNN is already fast per epoch. |
| Optimizer | **Adam** | The CNN extension confirmed Adam + lr=0.001 is the canonical choice for CNNs. SGD+Momentum won on ANNs, but adaptive rates matter more for conv layers. |
| Epochs | **5 (early stopping on val_loss, patience=2)** | CNN best val loss was at epoch 4 in the baseline and epoch 2–4 across LR variations. Training beyond that is strictly overfitting (train acc → 0.99, val loss → 0.64). |

In [ ]:
# ─── Part 4: Best Model ────────────────────────────────────────────────────────
# Rationale:
#   • CNN wins on val accuracy (~0.91) vs best ANN D6_very_deep (0.889)
#   • lr=0.001 + Adam is the best CNN configuration per the optional extension
#   • Early stopping at patience=2 on val_loss captures best generalisation
#     (CNN baseline: best val epoch = 4; across LR sweep: best between epochs 2–4)
#   • Batch size 256: good val loss in ANN experiments (0.38), stable gradients


best_model = tf.keras.Sequential([
    tf.keras.layers.Reshape((28, 28, 1), input_shape=(28, 28)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

best_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

best_model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True   # rolls back to best checkpoint automatically
)

history_best = best_model.fit(
    X_train, y_train,
    epochs=30,              # ceiling; early stopping will fire around epoch 4–6
    batch_size=256,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

plot_history(history_best, 'Best Model — CNN + EarlyStopping')
print("\n--- Best Model Results ---")
summarize_history(history_best)

### Test Set Evaluation

> **Warning:** Run this cell only ONCE. The test set must not be used for tuning.


In [ ]:
# ─── Test Set Evaluation — (run once) ───────────────────────────────────────────
test_loss, test_accuracy = best_model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')

### Confusion Matrix

In [ ]:
# ─── Confusion Matrix ─────────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = np.argmax(best_model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', values_format='d', xticks_rotation=45)
ax.set_title('Confusion Matrix — Final Model on Test Set')
plt.tight_layout()
plt.show()

# Most confused pairs
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
most_confused = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
print(f"\nMost confused pair: {class_names[most_confused[0]]} → predicted as {class_names[most_confused[1]]}")

#### Confusion Matrix Analysis

**TODO:** Which class pairs are most commonly confused? Hypothesize why based on visual similarity.

**Most confused pairs:**
- **T-shirt/top ↔ Shirt** (127 + 76 = 203 errors) — largest off-diagonal values by far; both short-sleeved upper-body garments, collar/placket distinction vanishes at 28×28.
- **Upper garment cluster** — Pullover→Shirt (88), Coat→Shirt (79), Shirt→Coat (52), Pullover→Coat (62); all share center-symmetric torso silhouette, fine details like lapels don't survive downsampling
- **Ankle boot ↔ Sneaker** (57 + 10 = 67 errors) — same thick sole, horizontal orientation; the asymmetry (more boots misclassified as sneakers than vice versa) suggests a slight Sneaker prediction bias at the shared decision boundary

**Clean classes:**
- Trouser (978), Bag (988), Sandal (964) — near-perfect; globally distinct shapes that are trivially separable even without spatial features

**Takeaway:** errors are entirely intra-category — tops confused with tops, footwear with footwear, never cross-category. A targeted second-stage classifier on the garment cluster would recover most of the remaining accuracy gap.


---
## 8. Summary of All Results

Run the cell below to generate a consolidated summary table of all experiments.


In [ ]:
# Consolidated results table
print("=" * 90)
print(f"{'Experiment':<30} {'Train Acc':>12} {'Val Acc':>12} {'Val Loss':>12} {'Gap':>12}")
print("=" * 90)

# Learning rate experiments
print("\n--- Learning Rate ---")
for lr, h in lr_histories.items():
    ta = h.history['accuracy'][-1]
    va = h.history['val_accuracy'][-1]
    vl = h.history['val_loss'][-1]
    print(f"  lr={lr:<22} {ta:>12.4f} {va:>12.4f} {vl:>12.4f} {ta-va:>12.4f}")

# Batch size experiments
print("\n--- Batch Size ---")
for bs, h in bs_histories.items():
    ta = h.history['accuracy'][-1]
    va = h.history['val_accuracy'][-1]
    vl = h.history['val_loss'][-1]
    print(f"  bs={bs:<23} {ta:>12.4f} {va:>12.4f} {vl:>12.4f} {ta-va:>12.4f}")

# Optimizer experiments
print("\n--- Optimizer ---")
for name, h in opt_histories.items():
    ta = h.history['accuracy'][-1]
    va = h.history['val_accuracy'][-1]
    vl = h.history['val_loss'][-1]
    print(f"  {name:<27} {ta:>12.4f} {va:>12.4f} {vl:>12.4f} {ta-va:>12.4f}")

# Architecture experiments
print("\n--- Architecture ---")
for name, h in arch_histories.items():
    ta = h.history['accuracy'][-1]
    va = h.history['val_accuracy'][-1]
    vl = h.history['val_loss'][-1]
    params = arch_params[name]
    print(f"  {name:<27} {ta:>12.4f} {va:>12.4f} {vl:>12.4f} {ta-va:>12.4f}  ({params:,} params)")

# CNN
print("\n--- CNN ---")
ta = history_cnn.history['accuracy'][-1]
va = history_cnn.history['val_accuracy'][-1]
vl = history_cnn.history['val_loss'][-1]
print(f"  {'CNN Baseline':<27} {ta:>12.4f} {va:>12.4f} {vl:>12.4f} {ta-va:>12.4f}")

print("\n" + "=" * 90)


---
## 9. Reflection

**1. What was the single most surprising finding from your experiments?**

- SGD+Momentum beat Adam on final val loss (0.3520 vs 0.4671), despite Adam converging faster and reaching lower train loss
- Expected adaptive optimizers to win — result suggests SGD+Momentum's noisier updates act as implicit regularization on a simple dataset like Fashion-MNIST
- Close second: lr=0.0001 achieved the best final val metrics of all five LRs (val loss 0.3331, val acc 0.8805) — usually dismissed as "too slow" but it outperformed every other rate within 30 epochs

---

**2. If you had unlimited compute time, what additional experiment would you run?**

- Joint sweep over LR schedules (cosine annealing, warm restarts) × early stopping patience — the optimal stopping point varied from epoch 2 (CNN lr=0.01) to epoch 29 (ANN lr=0.0001), so LR effect and epoch budget are currently confounded
- Extend batch size beyond bs=1024 — it showed the lowest val loss (0.3573) with no sharp-minima generalization penalty; would test bs=4096+ to see if the penalty eventually appears.

---

**3. What regularization technique would most improve your best model, and why?**

- **Data augmentation** (random horizontal flips + small crops)
- The CNN's val loss bottomed at 0.2436 (epoch 5) then climbed to 0.6558 by epoch 30 — classic memorization of training-set-specific spatial configurations
- Dropout reduces overfitting but doesn't improve representation quality; BatchNorm stabilizes training but convergence is already clean; data augmentation fixes the root cause
- Training set has every garment in a fixed orientation/crop — augmentation forces the model to learn genuinely translation- and reflection-invariant features, directly closing the 8.5pp train-val gap


---
## 10. Voluntary Extension: Optimiser Deep Dive

Part 2 compared four optimisers briefly. This section runs a broader set on the CNN architecture and connects each optimizer's  *internal mechanics* to its observed training behaviour.

**Optimizers tested:**
| Optimizer | Key mechanism |
|-----------|---------------|
| SGD | Vanilla gradient descent — no memory, no adaptation |
| SGD + Nesterov | Looks ahead: computes gradient at the *anticipated* next position |
| Adagrad | Per-parameter adaptive LR — divides by cumulative sum of squared gradients |
| RMSprop | Fixes Adagrad's decay problem — uses exponential moving average instead |
| Adam | RMSprop + momentum: adapts LR *and* smooths gradient direction |
| AdamW | Adam + decoupled weight decay — regularizes without distorting adaptive LR |
| Nadam | Adam + Nesterov look-ahead — faster convergence in theory |

All experiments use the CNN architecture, lr=0.001, batch size=64, 30 epochs.


In [ ]:
optimizer_extended = {
    'SGD':             tf.keras.optimizers.SGD(learning_rate=0.001),
    'SGD+Nesterov':    tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9, nesterov=True),
    'Adagrad':         tf.keras.optimizers.Adagrad(learning_rate=0.001),
    'RMSprop':         tf.keras.optimizers.RMSprop(learning_rate=0.001),
    'Adam':            tf.keras.optimizers.Adam(learning_rate=0.001),
    'AdamW':           tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4),
    'Nadam':           tf.keras.optimizers.Nadam(learning_rate=0.001),
}

ext_histories = {}

for name, opt in optimizer_extended.items():
    print(f'\n{"="*60}')
    print(f'Training CNN with optimizer: {name}')
    print(f'{"="*60}')

    model = build_cnn()
    model.compile(
        optimizer=opt,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train, y_train,
        epochs=30,
        batch_size=64,
        validation_data=(X_val, y_val),
        verbose=0
    )

    ext_histories[name] = history
    plot_history(history, f'CNN — {name}')
    summarize_history(history)


In [ ]:
plot_overlay(ext_histories, metric='val_loss',     title='CNN Optimizer Comparison — Validation Loss')
plot_overlay(ext_histories, metric='val_accuracy',  title='CNN Optimizer Comparison — Validation Accuracy')


In [ ]:
# Bar chart: best val loss per optimizer (at best val epoch)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(ext_histories.keys())
best_val_losses  = [min(ext_histories[n].history['val_loss'])     for n in names]
best_val_accs    = [max(ext_histories[n].history['val_accuracy']) for n in names]
best_val_epochs  = [np.argmin(ext_histories[n].history['val_loss']) + 1 for n in names]

colors = plt.cm.tab10.colors[:len(names)]

axes[0].barh(names, best_val_losses,  color=colors, edgecolor='white')
axes[0].set_xlabel('Best Validation Loss')
axes[0].set_title('Best Val Loss by Optimizer')
axes[0].grid(True, alpha=0.3, axis='x')
for i, (v, e) in enumerate(zip(best_val_losses, best_val_epochs)):
    axes[0].text(v + 0.003, i, f'{v:.4f} (ep {e})', va='center', fontsize=9)

axes[1].barh(names, best_val_accs, color=colors, edgecolor='white')
axes[1].set_xlabel('Best Validation Accuracy')
axes[1].set_title('Best Val Accuracy by Optimizer')
axes[1].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(best_val_accs):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()


#### Analysis: Optimizer Mechanics vs. Observed Behaviour

---

**SGD (vanilla)**
- *Mechanism:* pure gradient descent — every parameter moves by `lr × gradient`, with no memory of past updates and no per-parameter scaling
- *Expected behaviour:* slowest convergence, most stable curves (no momentum to overshoot), but high risk of getting stuck in flat regions
- *What to look for in the plot:* val loss still descending at epoch 30, no overfitting spike — but the lowest final accuracy of all seven

---

**SGD + Nesterov**
- *Mechanism:* instead of computing the gradient at the current position, Nesterov first takes the momentum step and then computes the gradient at that *anticipated* future position — effectively "correcting" the update before it overshoots
- *Expected behaviour:* faster convergence than vanilla SGD+Momentum because the look-ahead gradient is a better estimate of the direction after the step; less oscillation near minima
- *What to look for:* smoother val loss curve than SGD, faster initial descent than plain momentum, similar generalization ceiling

---

**Adagrad**
- *Mechanism:* accumulates the sum of all squared gradients per parameter and divides the learning rate by the square root of that sum — parameters that received large gradients historically get a smaller effective LR
- *Fatal flaw:* the denominator only ever grows, so the effective LR monotonically shrinks toward zero — the model eventually stops learning
- *Expected behaviour:* fast early convergence (large initial steps on sparse features), then hard plateau as the accumulated denominator kills all updates; distinctively flat val loss after the first few epochs

---

**RMSprop**
- *Mechanism:* fixes Adagrad's decaying LR by replacing the cumulative sum with an *exponential moving average* of squared gradients (controlled by decay rate ρ, default 0.9) — recent gradients matter more, old ones fade
- *Expected behaviour:* avoids Adagrad's plateau; fast convergence early on; but without a first-moment term, updates can be noisy in direction, leading to val loss drift in later epochs
- *What to look for:* good early descent, then val loss rising after best-val epoch — consistent with what Part 2 showed (best val epoch 9, then divergence)

---

**Adam**
- *Mechanism:* combines RMSprop (second moment — scales LR per parameter) with momentum (first moment — smooths gradient direction); both moments are bias-corrected in early epochs to avoid underestimation
- *Expected behaviour:* fast convergence, stable early curves; but tends to find sharper minima than SGD — val loss typically diverges after best-val epoch, more so than SGD+Momentum
- *What to look for:* quick drop to low val loss (~epoch 4–6), then steady upward creep — the classic Adam overfit signature on Fashion-MNIST

---

**AdamW**
- *Mechanism:* standard Adam applies L2 regularization by adding `λ × weight` to the gradient before the adaptive scaling — this means the regularization term itself gets scaled down by the adaptive denominator, making it weaker for parameters that matter most. AdamW *decouples* weight decay: it applies `λ × weight` as a direct weight shrinkage step *after* the Adam update, independent of the gradient scaling
- *Expected behaviour:* same convergence speed as Adam early on, but better generalization in later epochs — the decoupled weight decay acts as genuine regularization rather than a distorted gradient penalty; val loss divergence should be slower or shallower
- *What to look for:* val loss curves tracking Adam closely early, then AdamW pulling ahead (lower final val loss) as regularization kicks in

---

**Nadam**
- *Mechanism:* replaces Adam's standard momentum term with Nesterov momentum — instead of applying the accumulated gradient of the *previous* step, it applies the accumulated gradient of the *current anticipated* step. In practice this means the parameter update is computed at a slightly further-ahead position, giving a better gradient estimate
- *Expected behaviour:* marginally faster convergence than Adam (Nesterov look-ahead accelerates early descent), similar generalization ceiling; may overfit slightly faster because it finds minima quicker
- *What to look for:* val loss reaching its minimum 1–2 epochs earlier than Adam, then tracking a similar divergence trajectory

---

**Key takeaways**
- The SGD family (vanilla → momentum → Nesterov) trades speed for generalization: slower convergence, but flatter minima and lower tendency to overfit
- The adaptive family (Adagrad → RMSprop → Adam → AdamW/Nadam) trades generalization for speed: fast convergence, but sharper minima and earlier val loss divergence
- AdamW is the practical best-of-both-worlds attempt: adaptive convergence speed with genuine weight decay regularization — expected to outperform Adam on val loss while matching it on speed
- For CNNs on Fashion-MNIST specifically, the inductive bias of the architecture already provides strong regularization, so the optimizer choice mainly affects *when* to stop training rather than *whether* the model generalizes


---
*End of notebook. Remember to submit both this .ipynb file and your 5-page summary report (PDF).*
